In [108]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Dataset Prep
Only applicable for mockup to run temporary data pipeline. Need secure final pipeline from DSV.


In [109]:
randomiser = pd.read_excel("../data/Randomiser_True.xlsx")
randomiser.head()

,LL#,EXTERNORDERKEY - Order,HAWB,CARTONID,LPN = pallet ID,Pick & Pack Release: DEFAULTFORW,Cargo Release Ops: FINALFORW - Final Forwarder,ORDERCREATIONDATE,HANDOVERDATE - physical Handpover to forwarder
0,1097191,4037810554,SIN80245212,4037810554001,DCA200000011166,SCHENKER SINGAPORE (PTE) LTD,SCHENKER SINGAPORE (PTE) LTDLocal Del -2nd leg,2026-03-26 10:51:47,2026-04-07 17:08:31
1,1097191,4037826956,SIN80245212,4037826956001,DCA200000050296,SCHENKER SINGAPORE (PTE) LTD,SCHENKER SINGAPORE (PTE) LTDLocal Del -2nd leg,2026-03-27 06:53:56,2026-04-07 17:08:33
2,1117194,4037862826,4942047235,4037862826001,DCA200000049614,DIMERCO EXPRESS PTE LTD.,DHL Express (Singapore) Pte Ltd -INTERNATIONAL,2026-04-07 13:51:49,2026-04-07 18:30:58
3,1123987,4037876046,SIN69803196,4037876046002,DCA200000011922,JAS FORWARDING (S) PTE LTD,JAS FORWARDING (S) PTE LTD,2026-04-10 14:00:46,2026-04-10 17:01:54
4,1123987,4037876046,SIN69803196,4037876046002,DCA200000011922,JAS FORWARDING (S) PTE LTD,JAS FORWARDING (S) PTE LTD,2026-04-10 14:00:46,2026-04-10 17:01:54


In [110]:
print(randomiser["LL#"].nunique())
print(randomiser['HANDOVERDATE - physical Handpover to forwarder'].nunique())

23
104


In [111]:
# drop columns that is not handoverdate and pick and pack release
randomiser = randomiser[['Pick & Pack Release: DEFAULTFORW', 'HANDOVERDATE - physical Handpover to forwarder']]
randomiser = randomiser.rename(columns={'Pick & Pack Release: DEFAULTFORW': 'Forwarder', 'HANDOVERDATE - physical Handpover to forwarder': 'Handover Date'})

In [112]:
number_forwereders = randomiser["Forwarder"].nunique()
print(f"Number of unique forwarders: {number_forwereders}")

Number of unique forwarders: 12


In [113]:
# get the time distribution of handover time in minutes relative to 00:00 on the day of handover
randomiser['Time of Handover'] = (randomiser['Handover Date'] - randomiser['Handover Date'].dt.normalize()).dt.total_seconds() / 60
mean_time = randomiser['Time of Handover'].mean()
std_time = randomiser['Time of Handover'].std()
print(f"Mean time of handover: {mean_time:.2f} minutes")
print(f"Standard deviation of time of handover: {std_time:.2f} minutes")

Mean time of handover: 1048.84 minutes
Standard deviation of time of handover: 130.75 minutes


### Primary Data for Dashboard Simulation


In [114]:
data = pd.read_excel("../data/Test_True.xlsx")
data.head()

,Shipping TU ID,DN,Shipped To,Check Out Date,Create Date,Complete Date,Created By,dispatchWorkstationId,Line Number,Loading List,No. of carton,Loading List.1,Loading List Line Key,sendToDispatchDate,Status,Transaction Count,Update Date,Updated By,Location,Location.1
0,4038186026001,4038186026,JIANGSU HIRAIN AUTOMOTIVE,2026-09-08 14:55:24.384,2026-09-08 14:40:18.702,2026-09-08 14:57:59.369,HOST,OB2I,14,1308267,15,286569,1123475,2026-09-08 14:54:56.807,BUFFER,8,2026-09-08 14:55:24.475,Murphy,OB2I_WORK,NaN
1,4038186026002,4038186026,JIANGSU HIRAIN AUTOMOTIVE,2026-09-08 14:54:28.414,2026-09-08 14:40:18.702,2026-09-08 14:57:59.369,HOST,OB2I,13,1308267,15,286569,1123474,2026-09-08 14:42:57.337,BUFFER,8,2026-09-08 14:54:28.493,Murphy,OB2I_WORK,NaN
2,4038186026003,4038186026,JIANGSU HIRAIN AUTOMOTIVE,2026-09-08 14:54:18.652,2026-09-08 14:40:18.702,2026-09-08 14:57:59.369,HOST,OB2I,12,1308267,15,286569,1123473,2026-09-08 14:42:53.967,BUFFER,8,2026-09-08 14:54:18.731,Murphy,OB2I_WORK,NaN
3,4038186026004,4038186026,JIANGSU HIRAIN AUTOMOTIVE,2026-09-08 14:54:55.353,2026-09-08 14:40:18.702,2026-09-08 14:57:59.369,HOST,OB2I,11,1308267,15,286569,1123472,2026-09-08 14:43:25.280,BUFFER,8,2026-09-08 14:54:55.461,Murphy,OB2I_WORK,NaN
4,4038186026005,4038186026,JIANGSU HIRAIN AUTOMOTIVE,2026-09-08 14:55:37.779,2026-09-08 14:40:18.702,2026-09-08 14:57:59.369,HOST,OB2I,10,1308267,15,286569,1123471,2026-09-08 14:55:04.260,BUFFER,8,2026-09-08 14:55:37.847,Murphy,OB2I_WORK,NaN


In [115]:
# DN -> Unique ID for Loading List
# Cartons -> No. of Cartons in LL
# Create Date -> Time LL added to SynQ
# Complete Date -> Time LL completed in SynQ
# Zone Data -> 48 locations across 6 zones, 288 locations in total
test_data = data[['Loading List', 'No. of carton', 'Create Date', 'Complete Date']].copy()
test_data.rename(columns={'No. of carton': 'Cartons', 'Complete Date': 'Complete Date', 'Create Date': 'Added Date'}, inplace=True)
test_data.sort_values(by='Cartons', ascending = False, inplace=True)
test_data.drop_duplicates(subset='Loading List', keep='first', inplace=True)
test_data.reset_index(drop=True, inplace=True)
print(f"Number of unique loading lists: {test_data['Loading List'].nunique()}")
print(f"Number of original loading lists: {data['Loading List'].nunique()}")

Number of unique loading lists: 327
Number of original loading lists: 327


C:\Users\choon\AppData\Local\Temp\ipykernel_14624\2963534065.py:7: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  test_data.rename(columns={'No. of carton': 'Cartons', 'Complete Date': 'Complete Date', 'Create Date': 'Added Date'}, inplace=True)
C:\Users\choon\AppData\Local\Temp\ipykernel_14624\2963534065.py:8: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  test_data.sort_values(by='Cartons', ascending = False, inplace=True)
C:\Users\choon\AppData\Local\Temp\ipykernel_14624\2963534065.py:9: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydat

In [116]:
test_data.head()

,Loading List,Cartons,Added Date,Complete Date
0,1308131,205,2026-09-08 06:38:32.198,2026-09-08 07:26:55.755
1,1307912,197,2026-09-08 10:58:32.640,2026-09-08 12:19:57.636
2,1308234,192,2026-09-08 15:15:33.240,2026-09-08 16:48:01.362
3,1307871,182,2026-09-08 09:41:32.061,2026-09-08 10:43:47.443
4,1307695,152,2026-09-08 11:21:47.724,2026-09-08 13:36:45.200


### Zoning and Handover Table
Forwarders A-L assigned uniformly at random; zones 1-6 hold two forwarders each. 1 pallet per 50 cartons, rounded up (minimum 1). Handover time drawn from the observed distribution, floored at Added Date + 1h.

In [117]:
rng = np.random.default_rng(42)

ll_table = test_data.rename(columns={'Loading List': 'LL'}).sort_values('Added Date').reset_index(drop=True)

# 1. Uniform random forwarder, 12 forwarders labelled A-L
forwarders = list("ABCDEFGHIJKL")
ll_table['Forwarder'] = rng.choice(forwarders, size=len(ll_table))

# 2. Every 2 forwarders share a zone: A,B -> 1 ; C,D -> 2 ; ... ; K,L -> 6
zone_map = {f: i // 2 + 1 for i, f in enumerate(forwarders)}
ll_table['Zone'] = ll_table['Forwarder'].map(zone_map)

# 3. One pallet per 50 cartons (rounded up), minimum one pallet per LL
ll_table['Pallets'] = np.ceil(ll_table['Cartons'] / 50).clip(lower=1).astype(int)

# 4. Handover time sampled from the observed handover time-of-day distribution (normal)
sampled_minutes = rng.normal(mean_time, std_time, size=len(ll_table))
handover = ll_table['Added Date'].dt.normalize() + pd.to_timedelta(sampled_minutes, unit='m')
# If handover falls before the LL was added, use Added Date + 1 hour
too_early = handover < ll_table['Added Date']
ll_table['Handover Time'] = handover.where(~too_early, ll_table['Added Date'] + pd.Timedelta(hours=1))

print(f"LLs: {len(ll_table)}, handover moved to Added Date + 1h for {too_early.sum()} LLs")
print(ll_table.groupby('Zone').agg(LLs=('LL', 'count'), Pallets=('Pallets', 'sum')))
ll_table.head()

LLs: 327, handover moved to Added Date + 1h for 9 LLs
      LLs  Pallets
Zone              
1      54       57
2      51       58
3      63       69
4      52       57
5      68       89
6      39       42


,LL,Cartons,Added Date,Complete Date,Forwarder,Zone,Pallets,Handover Time
0,1308118,20,2026-09-08 06:24:02.111,2026-09-08 06:36:44.898,B,1,1,2026-09-08 15:54:01.124248182
1,1308120,5,2026-09-08 06:24:02.687,2026-09-08 06:37:36.047,J,5,1,2026-09-08 22:07:07.803990768
2,1308121,1,2026-09-08 06:24:03.278,2026-09-08 06:28:17.020,H,4,1,2026-09-08 15:41:26.763223728
3,1308166,1,2026-09-08 06:24:03.825,2026-09-08 06:27:50.331,F,3,1,2026-09-08 19:18:28.161595524
4,1308165,9,2026-09-08 06:24:04.384,2026-09-08 06:29:23.990,F,3,1,2026-09-08 15:30:47.094119532


In [118]:
ll_table.to_excel("../data/Mockup_Data.xlsx", index=False)